<a href="https://colab.research.google.com/github/KMKalingavelan/daa/blob/main/daa8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Travelling Salesman Problem (TSP)
# Branch and Bound using Reduced Cost Matrix
# 5 Cities


import math
import heapq


# ==========================================================
# COST MATRIX
# ==========================================================

# 0, 1, 2, 3, 4 represent the five cities
#
# INF means a city cannot travel to itself.

INF = math.inf

cost_matrix = [
    [INF, 20, 30, 10, 11],
    [15, INF, 16, 4, 2],
    [3, 5, INF, 2, 4],
    [19, 6, 18, INF, 3],
    [16, 4, 7, 16, INF]
]

N = len(cost_matrix)


# ==========================================================
# REDUCE COST MATRIX
# ==========================================================

def reduce_matrix(matrix):

    matrix = [row[:] for row in matrix]

    reduction_cost = 0

    # ------------------------------------------------------
    # Row reduction
    # ------------------------------------------------------

    for i in range(N):

        row_min = min(matrix[i])

        if row_min != INF and row_min > 0:

            reduction_cost += row_min

            for j in range(N):

                if matrix[i][j] != INF:
                    matrix[i][j] -= row_min


    # ------------------------------------------------------
    # Column reduction
    # ------------------------------------------------------

    for j in range(N):

        col_min = min(matrix[i][j] for i in range(N))

        if col_min != INF and col_min > 0:

            reduction_cost += col_min

            for i in range(N):

                if matrix[i][j] != INF:
                    matrix[i][j] -= col_min


    return matrix, reduction_cost


# ==========================================================
# CREATE CHILD NODE
# ==========================================================

def create_child(parent_matrix, from_city, to_city):

    matrix = [row[:] for row in parent_matrix]

    # Block the entire row of the source city
    for j in range(N):
        matrix[from_city][j] = INF

    # Block the entire column of the destination city
    for i in range(N):
        matrix[i][to_city] = INF

    # Prevent returning to the starting city too early
    matrix[to_city][0] = INF

    return matrix


# ==========================================================
# BRANCH AND BOUND TSP
# ==========================================================

def tsp_branch_and_bound():

    # Reduce the original matrix
    reduced_matrix, initial_bound = reduce_matrix(cost_matrix)

    # Priority queue:
    # (lower_bound, city, path, matrix)

    priority_queue = []

    heapq.heappush(
        priority_queue,
        (initial_bound, 0, [0], reduced_matrix)
    )

    best_cost = INF
    best_path = None

    while priority_queue:

        bound, current_city, path, matrix = heapq.heappop(
            priority_queue
        )

        # Prune if lower bound is already worse
        if bound >= best_cost:
            continue

        # --------------------------------------------------
        # All cities visited
        # --------------------------------------------------

        if len(path) == N:

            return_cost = cost_matrix[current_city][0]

            if return_cost != INF:

                total_cost = bound + return_cost

                if total_cost < best_cost:

                    best_cost = total_cost
                    best_path = path + [0]

            continue


        # --------------------------------------------------
        # Branch to unvisited cities
        # --------------------------------------------------

        for next_city in range(N):

            if next_city in path:
                continue

            original_cost = cost_matrix[current_city][next_city]

            if original_cost == INF:
                continue

            child_matrix = create_child(
                matrix,
                current_city,
                next_city
            )

            # Reduce child matrix
            reduced_child, reduction_cost = reduce_matrix(
                child_matrix
            )

            # Calculate lower bound
            child_bound = (
                bound
                + original_cost
                + reduction_cost
            )

            # Branch only if promising
            if child_bound < best_cost:

                heapq.heappush(
                    priority_queue,
                    (
                        child_bound,
                        next_city,
                        path + [next_city],
                        reduced_child
                    )
                )

    return best_cost, best_path


# ==========================================================
# RUN ALGORITHM
# ==========================================================

best_cost, best_path = tsp_branch_and_bound()


# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("Travelling Salesman Problem")
print("===========================")

print("\nCost Matrix:")

for row in cost_matrix:
    print(row)

print("\nOptimal Tour:")

for i, city in enumerate(best_path):

    if i < len(best_path) - 1:
        print(city, end=" -> ")
    else:
        print(city)

print("\n\nMinimum Tour Cost:", best_cost)

Travelling Salesman Problem

Cost Matrix:
[inf, 20, 30, 10, 11]
[15, inf, 16, 4, 2]
[3, 5, inf, 2, 4]
[19, 6, 18, inf, 3]
[16, 4, 7, 16, inf]

Optimal Tour:
0 -> 3 -> 1 -> 4 -> 2 -> 0


Minimum Tour Cost: 53


In [4]:
# ==========================================================
# TSP: Brute Force vs Nearest Neighbour
# 6-City City Grid
# Cost = Road Distance in km
# ==========================================================

import itertools
import time


# ----------------------------------------------------------
# Distance Matrix
# ----------------------------------------------------------

# Cities:
# 0 = A
# 1 = B
# 2 = C
# 3 = D
# 4 = E
# 5 = F

cities = ["A", "B", "C", "D", "E", "F"]

distance = [
    [0, 10, 15, 20, 25, 18],
    [10, 0, 35, 25, 17, 28],
    [15, 35, 0, 30, 20, 22],
    [20, 25, 30, 0, 15, 12],
    [25, 17, 20, 15, 0, 10],
    [18, 28, 22, 12, 10, 0]
]

N = len(cities)


# ==========================================================
# 1. BRUTE FORCE TSP
# ==========================================================

def tsp_brute_force():

    start_city = 0

    best_distance = float("inf")
    best_route = None

    # Generate every possible permutation
    for permutation in itertools.permutations(
        range(1, N)
    ):

        route = (start_city,) + permutation + (start_city,)

        total_distance = 0

        # Calculate route distance
        for i in range(len(route) - 1):

            total_distance += distance[
                route[i]
            ][
                route[i + 1]
            ]

        # Update best route
        if total_distance < best_distance:

            best_distance = total_distance
            best_route = route

    return best_route, best_distance


# ==========================================================
# 2. NEAREST NEIGHBOUR HEURISTIC
# ==========================================================

def nearest_neighbour():

    start_city = 0

    visited = {start_city}

    route = [start_city]

    current_city = start_city

    total_distance = 0

    # Visit nearest unvisited city
    while len(visited) < N:

        nearest_city = None
        nearest_distance = float("inf")

        for city in range(N):

            if city not in visited:

                if distance[current_city][city] < nearest_distance:

                    nearest_distance = distance[
                        current_city
                    ][
                        city
                    ]

                    nearest_city = city

        # Move to nearest city
        route.append(nearest_city)

        visited.add(nearest_city)

        total_distance += nearest_distance

        current_city = nearest_city

    # Return to starting city
    route.append(start_city)

    total_distance += distance[
        current_city
    ][
        start_city
    ]

    return route, total_distance


# ==========================================================
# 3. RUN BRUTE FORCE
# ==========================================================

start_time = time.perf_counter()

brute_route, brute_distance = tsp_brute_force()

brute_time = time.perf_counter() - start_time


# ==========================================================
# 4. RUN NEAREST NEIGHBOUR
# ==========================================================

start_time = time.perf_counter()

nn_route, nn_distance = nearest_neighbour()

nn_time = time.perf_counter() - start_time


# ==========================================================
# 5. Convert route numbers to city names
# ==========================================================

def format_route(route):

    return " -> ".join(
        cities[city] for city in route
    )


# ==========================================================
# 6. DISPLAY RESULTS
# ==========================================================

print("TRAVELLING SALESMAN PROBLEM")
print("============================")

print("\nCities:")
print("A, B, C, D, E, F")

print("\nDistance Matrix (km):")

for row in distance:
    print(row)


# ----------------------------------------------------------
# Brute Force Result
# ----------------------------------------------------------

print("\n\nBRUTE FORCE")
print("-----------")

print("Optimal Route:")
print(format_route(brute_route))

print("Total Distance:",
      brute_distance, "km")

print("Execution Time:",
      brute_time, "seconds")


# ----------------------------------------------------------
# Nearest Neighbour Result
# ----------------------------------------------------------

print("\n\nNEAREST NEIGHBOUR")
print("-----------------")

print("Route:")
print(format_route(nn_route))

print("Total Distance:",
      nn_distance, "km")

print("Execution Time:",
      nn_time, "seconds")


# ==========================================================
# 7. COMPARISON
# ==========================================================

difference = nn_distance - brute_distance

percentage_difference = (
    difference / brute_distance
) * 100


print("\n\nCOMPARISON")
print("----------")

print("Optimal Distance (Brute Force):",
      brute_distance, "km")

print("Nearest Neighbour Distance:",
      nn_distance, "km")

print("Extra Distance:",
      difference, "km")

print("Percentage Difference:",
      round(percentage_difference, 2), "%")


if nn_distance == brute_distance:

    print("\nNearest Neighbour found the optimal route.")

else:

    print("\nNearest Neighbour did NOT find the optimal route.")

TRAVELLING SALESMAN PROBLEM

Cities:
A, B, C, D, E, F

Distance Matrix (km):
[0, 10, 15, 20, 25, 18]
[10, 0, 35, 25, 17, 28]
[15, 35, 0, 30, 20, 22]
[20, 25, 30, 0, 15, 12]
[25, 17, 20, 15, 0, 10]
[18, 28, 22, 12, 10, 0]


BRUTE FORCE
-----------
Optimal Route:
A -> B -> E -> D -> F -> C -> A
Total Distance: 91 km
Execution Time: 0.00015975299999126946 seconds


NEAREST NEIGHBOUR
-----------------
Route:
A -> B -> E -> F -> D -> C -> A
Total Distance: 94 km
Execution Time: 9.155200001487174e-05 seconds


COMPARISON
----------
Optimal Distance (Brute Force): 91 km
Nearest Neighbour Distance: 94 km
Extra Distance: 3 km
Percentage Difference: 3.3 %

Nearest Neighbour did NOT find the optimal route.


In [3]:
# ==========================================================
# TSP: 2-Opt Improvement Heuristic
# 10-City Random Instance
#
# Steps:
# 1. Generate random distance matrix
# 2. Find Nearest-Neighbour tour
# 3. Improve using 2-Opt
# 4. Find optimal solution using Brute Force
# 5. Compare 2-Opt with optimal solution
# ==========================================================

import random
import itertools
import time


# ----------------------------------------------------------
# 1. Generate Random 10-City Distance Matrix
# ----------------------------------------------------------

random.seed(42)

N = 10

cities = [chr(ord('A') + i) for i in range(N)]

distance = [[0] * N for _ in range(N)]

for i in range(N):
    for j in range(i + 1, N):

        d = random.randint(10, 100)

        distance[i][j] = d
        distance[j][i] = d


# ----------------------------------------------------------
# Calculate total tour distance
# ----------------------------------------------------------

def tour_distance(tour):

    total = 0

    for i in range(len(tour) - 1):
        total += distance[
            tour[i]
        ][
            tour[i + 1]
        ]

    return total


# ==========================================================
# 2. Nearest-Neighbour Heuristic
# ==========================================================

def nearest_neighbour(start=0):

    visited = {start}

    tour = [start]

    current = start

    while len(tour) < N:

        nearest = None
        nearest_distance = float("inf")

        for city in range(N):

            if city not in visited:

                if distance[current][city] < nearest_distance:

                    nearest_distance = distance[
                        current
                    ][
                        city
                    ]

                    nearest = city

        tour.append(nearest)

        visited.add(nearest)

        current = nearest

    # Return to starting city
    tour.append(start)

    return tour


# ==========================================================
# 3. 2-OPT Improvement
# ==========================================================

def two_opt(tour):

    best_tour = tour[:]

    best_distance = tour_distance(best_tour)

    improvement = True

    iterations = 0

    while improvement:

        improvement = False

        # Try every possible segment reversal
        for i in range(1, N - 1):

            for j in range(i + 1, N):

                # Reverse the segment
                new_tour = (
                    best_tour[:i]
                    + best_tour[i:j + 1][::-1]
                    + best_tour[j + 1:]
                )

                new_distance = tour_distance(new_tour)

                # Accept improvement
                if new_distance < best_distance:

                    best_tour = new_tour

                    best_distance = new_distance

                    improvement = True

                    iterations += 1

                    break

            if improvement:
                break

    return best_tour, best_distance, iterations


# ==========================================================
# 4. Brute Force Optimal Solution
# ==========================================================

def brute_force_tsp():

    start = 0

    best_t